In [ ]:
import cProfile
import pstats
import io

def profile_function(func):
    """Decorator to profile a function."""
    def wrapper(*args, **kwargs):
        pr = cProfile.Profile()
        pr.enable()
        result = func(*args, **kwargs)
        pr.disable()
        s = io.StringIO()
        sortby = 'cumulative'
        ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
        ps.print_stats()
        print(s.getvalue())
        return result
    return wrapper


In [ ]:
import pandas as pd
import pandas_ta as ta
import pyotp
from datetime import datetime, timedelta
import time
import csv
import asyncio
import nest_asyncio
import logging
import websocket as ws_client
import xarray as xr
import threading

from NorenRestApiPy.NorenApi import NorenApi

class ShoonyaApiPy(NorenApi):
    def __init__(self):
        NorenApi.__init__(self, host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')        
        global api
        api = self

import logging
#logging.basicConfig(level=logging.DEBUG)
logging.getLogger('websocket').setLevel(logging.INFO)

usercred = pd.read_excel("usercred.xlsx")
user    = usercred.iloc[0,0]
pwd     = usercred.iloc[0,1]
vc      = usercred.iloc[0,2]
app_key = usercred.iloc[0,3]
imei    = usercred.iloc[0,4]
qr = usercred.iloc[0,5]
factor2 = pyotp.TOTP(qr).now()

api = ShoonyaApiPy()
ret = api.login(userid=user, password=pwd, twoFA=factor2, vendor_code=vc, api_secret=app_key, imei=imei)

In [ ]:
import pandas as pd
fno_scrips_mcx = pd.read_csv('//home/deep/Desktop/NEWshoonya/MCX_symbols.txt.zip',compression='zip', engine='python',delimiter=',')
fno_scrips_mcx['Expiry'] = pd.to_datetime(fno_scrips_mcx['Expiry'])
fno_scrips_mcx['StrikePrice'] = fno_scrips_mcx['StrikePrice'].astype(float)
fno_scrips_mcx.sort_values('Expiry',inplace=True)
fno_scrips_mcx.reset_index(drop=True, inplace=True)

fno_scrips = pd.read_csv('/home/deep/Desktop/NEWshoonya/NFO_symbols.txt.zip',compression='zip', engine='python',delimiter=',')
fno_scrips['Expiry'] = pd.to_datetime(fno_scrips['Expiry'])
fno_scrips['StrikePrice'] = fno_scrips['StrikePrice'].astype(float)
fno_scrips.sort_values('Expiry',inplace=True)
fno_scrips.reset_index(drop=True, inplace=True)

In [ ]:
import numpy as np
from numba import njit

@njit
def ema_numba(close, length):
    out = np.full_like(close, np.nan)
    
    if len(close) <= length:
        return out
    
    alpha = 2 / (length + 1)
    ema = np.mean(close[:length])  # Initialize with SMA
    out[length-1] = ema
    
    for i in range(length, len(close)):
        ema = alpha * close[i] + (1 - alpha) * ema
        out[i] = ema
    
    return np.round(out, 2)

import numpy as np
from numba import njit

@njit
def jma_numba(src, length, phase, power):
    out = np.full_like(src, np.nan)
    
    if len(src) <= length:
        return out
    
    phaseRatio = 1.5 + phase / 100 if -100 <= phase <= 100 else (0.5 if phase < -100 else 2.5)
    beta = 0.45 * (length - 1) / (0.45 * (length - 1) + 2)
    alpha = np.power(beta, power)
    
    e0 = e1 = e2 = jma = 0.0
    
    for i in range(len(src)):
        if i == 0:
            e0 = src[i]
            e1 = 0.0
            e2 = 0.0
            jma = 0.0
        else:
            e0 = (1 - alpha) * src[i] + alpha * e0
            e1 = (src[i] - e0) * (1 - beta) + beta * e1
            e2 = (e0 + phaseRatio * e1 - jma) * np.power(1 - alpha, 2) + np.power(alpha, 2) * e2
            jma = e2 + jma
            
            # Reset values if they become too large or unstable
            if np.abs(jma) > 1e10:
                e0 = e1 = e2 = jma = 0.0
        
        if i >= length - 1:
            out[i] = np.round(jma, 2)
    
    return out

import numpy as np
from numba import njit

@njit
def alma_numba(series, length, offset, sigma):
    out = np.full_like(series, np.nan)
    
    if len(series) < length:
        return out
    
    for i in range(length - 1, len(series)):
        numerator = 0.0
        denominator = 0.0
        m = offset * (length - 1)
        s = length / sigma
        for j in range(length):
            weight = np.exp(-((j - m) ** 2) / (2 * s * s))
            numerator += weight * series[i - length + 1 + j]
            denominator += weight
        out[i] = round(numerator / denominator, 2)
    
    return out


@njit
def rsi_trail_numba(open, high, low, close, rsi_lower=45, rsi_upper=55, ma_length=20, ma_offset=0.85, ma_sigma=6):
    ohlc4 = (open + high + low + close) / 4
    ma = alma_numba(ohlc4, ma_length, ma_offset, ma_sigma)
    atr = atr_numba(high, low, close, 7)
    upper_bound = ma + (rsi_upper - 50) / 10 * atr
    lower_bound = ma - (50 - rsi_lower) / 10 * atr
    
    signal = np.zeros_like(close, dtype=np.float64)
    is_bullish = np.zeros_like(close, dtype=np.bool_)
    is_bearish = np.zeros_like(close, dtype=np.bool_)
    
    signal[:ma_length] = np.nan  # Set initial values to NaN
        
    for i in range(ma_length, len(close)):
        if ohlc4[i] > upper_bound[i]:
            if not is_bullish[i-1]:
                signal[i] = 1  # Bullish crossover
            is_bullish[i] = True
            is_bearish[i] = False
        elif close[i] < lower_bound[i]:
            if not is_bearish[i-1]:
                signal[i] = -1  # Bearish crossunder
            is_bullish[i] = False
            is_bearish[i] = True
        else:
            signal[i] = 0  # Neutral
            is_bullish[i] = is_bullish[i-1]
            is_bearish[i] = is_bearish[i-1]
    
    return signal, is_bullish, is_bearish

@njit
def atr_numba(high, low, close, length):
    true_range = np.zeros_like(close)
    atr = np.full_like(close, np.nan)
    
    for i in range(1, len(close)):
        true_range[i] = max(high[i] - low[i], abs(high[i] - close[i-1]), abs(low[i] - close[i-1]))
    
    for i in range(length, len(close)):
        if i == length:
            atr[i] = np.mean(true_range[:length])
        else:
            atr[i] = (atr[i - 1] * (length - 1) + true_range[i]) / length
    
    return atr


@njit
def chandelier_exit_numba(open, high, low, close, length=2, mult=1):
    atr = atr_numba(high, low, close, length) * mult
    
    long_stop = np.full_like(close, np.nan)
    short_stop = np.full_like(close, np.nan)
    direction = np.zeros_like(close)
    
    for i in range(length, len(close)):
        highest = np.max(close[i-length+1:i+1])
        lowest = np.min(close[i-length+1:i+1])
        
        long_stop[i] = highest - atr[i]
        short_stop[i] = lowest + atr[i]
        
        if i > length:
            long_stop[i] = max(long_stop[i], long_stop[i-1]) if close[i-1] > long_stop[i-1] else long_stop[i]
            short_stop[i] = min(short_stop[i], short_stop[i-1]) if close[i-1] < short_stop[i-1] else short_stop[i]
        
        if close[i] > short_stop[i-1]:
            direction[i] = 1
        elif close[i] < long_stop[i-1]:
            direction[i] = -1
        else:
            direction[i] = direction[i-1] if i > 0 else 0
    
    return long_stop, short_stop, direction

@njit
def calculate_average_height_range(high, low, length):
    # Calculate the length of each bar
    last_10_high = high[-length:]
    last_10_low = low[-length:]

    current_high = high[-1:]
    currenn_low = low[-1:]

    # Calculate the height of each candle (high - low)
    avg_candle_heights = last_10_high - last_10_low
    current_candle_height = current_high - currenn_low
    average_height = np.mean(avg_candle_heights)
    
    return average_height, current_candle_height
    

@njit
def calculate_current_bar_length(high, low):
    # Calculate the length of the current bar
    current_bar_length = high[-1] - low[-1]
    return current_bar_length

In [ ]:
import asyncio
import threading
from datetime import datetime
import pandas as pd
import nest_asyncio
import time
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict, deque

feed_lock = threading.Lock()
range_length = 10
ema_length = 22
last_processed_candle = {}
trading_active = True
feed_opened = False
feedJson = defaultdict(lambda: deque(maxlen=10))  # Deque with maxlen=10 for each token
extra_feedJson = defaultdict(lambda: deque(maxlen=10))
resample_frequency = '5s'
resampling_lock = asyncio.Lock()
resampling_period_end = {}
last_resample_time = {}
resampled_data = {}
current_positions = {}
# Dictionary to keep track of entry instruments
entry_instruments = {}
entry_prices = {}
profit_target = 100
stop_loss = 20
bar_length_multiplier = 1.5
indicator_state = {}
finalized_data = {}
Exchange = 'MCX'
Initial_token = '428870'
subscription_string = Exchange + '|' + Initial_token
feed_lock = threading.Lock()
global signal_bar_high , signal_bar_low 
from concurrent.futures import ThreadPoolExecutor
trading_logic_task = None


def event_handler_feed_update(tick_data):
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']

            with feed_lock:
                if token == Initial_token:
                    # Append to the deque for the main token
                    feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
                
                else:
                    # Append to the deque for extra tokens
                    extra_feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})

    except (KeyError, ValueError) as e:
        print(f"Error processing tick data: {e}")

async def resample_ticks():
    global finalized_data
    while True:
        if not feedJson:
            await asyncio.sleep(0.01)  # Sleep for a short interval if no data
            continue

        async with resampling_lock:
            for token, ticks in feedJson.items():
                if not ticks:
                    continue

                try:
                    # Create a DataFrame with the new ticks
                    df_new = pd.DataFrame(ticks)
                    df_new["tt"] = pd.to_datetime(df_new["tt"])
                    df_new.set_index("tt", inplace=True)

                    # Initialize or update the resampled DataFrame
                    if token not in resampled_data:
                        # Initialize the DataFrame with the first resample
                        resampled_data[token] = df_new['ltp'].resample(resample_frequency).ohlc()
                        last_resample_time[token] = df_new.index.max()
                    else:
                        # Resample the new ticks
                        df_resampled = df_new['ltp'].resample(resample_frequency).ohlc()

                        # Update the existing resampled DataFrame with new data
                        for idx, row in df_resampled.iterrows():
                            if idx in resampled_data[token].index:
                                resampled_data[token].at[idx, 'high'] = max(resampled_data[token].at[idx, 'high'], row['high'])
                                resampled_data[token].at[idx, 'low'] = min(resampled_data[token].at[idx, 'low'], row['low'])
                                resampled_data[token].at[idx, 'close'] = row['close']
                            else:
                                resampled_data[token].loc[idx] = row
                        

                        # Update the last resampled time
                        last_resample_time[token] = df_new.index.max()
                    
                    

                    if token in resampled_data:
                        long_stop, short_stop, direction = chandelier_exit_numba(resampled_data[token]['open'].values,resampled_data[token]['high'].values,resampled_data[token]['low'].values,resampled_data[token]['close'].values)

                        resampled_data[token]['long_stop'] = long_stop
                        resampled_data[token]['short_stop'] = short_stop
                        resampled_data[token]['ce_direction'] = direction
                        
                    resampled_data[token] = resampled_data[token].dropna(subset=['open', 'high', 'low', 'close'])

                except Exception as e:
                    if isinstance(e, KeyError) and e.args[0] == 'tt':
                        print(f"Market likely closed for token {token}")
                    else:
                        print(f"Error resampling data for token {token}: {e}")

        await asyncio.sleep(0.01)  # Sleep for a short interval to avoid busy-waiting

def event_handler_order_update(tick_data):
    print(f"Order update {tick_data}")

def open_callback():
    global feed_opened
    feed_opened = True

# Assuming `api` is defined and starts the WebSocket connection
async def connect_and_subscribe():
    global feed_opened
    api.start_websocket(
        order_update_callback=event_handler_order_update,
        subscribe_callback=event_handler_feed_update,
        socket_open_callback=open_callback
    )
    while not feed_opened:
        await asyncio.sleep(1)  # Wait for initial connection
    api.subscribe([subscription_string])


def set_trading_active(value):
    global trading_active, trading_logic_task
    trading_active = value
    print(f"Trading active set to {trading_active}")

    if trading_active:
        if trading_logic_task is None or trading_logic_task.done():
            trading_logic_task = asyncio.create_task(trading_logic())
            print("Started trading logic task")
    else:
        if trading_logic_task is not None and not trading_logic_task.done():
            trading_logic_task.cancel()
            print("Cancelled trading logic task")


async def main():

    global trading_logic_task
   
    resample_task = asyncio.create_task(resample_ticks())
    connect_task = asyncio.create_task(connect_and_subscribe()) 
    trading_logic_task = asyncio.create_task(trading_logic())     
    update_symbols_task = asyncio.create_task(update_atm_symbols())
    
    await asyncio.gather(connect_task, resample_task, trading_logic_task, update_symbols_task)

# Get the current event loop
loop = asyncio.get_event_loop()

if loop.is_running():
    nest_asyncio.apply()
    asyncio.create_task(main())
else:
    loop.run_until_complete(main())

In [ ]:
async def trading_logic():
    
    global last_processed_candle, resampled_data, trading_active
    while trading_active:
        current_time = pd.Timestamp.now()        
        
        async with resampling_lock:
            for token, data in resampled_data.items():
                if not data.empty:
                    last_candle_start = data.index[-1]
                    resample_freq = data.index.freq or pd.Timedelta(seconds=5)
                    current_candle_end = last_candle_start + resample_freq

                    # Check if the current time is beyond the current candle's end time
                    if current_time >= current_candle_end:
                        # Ensure we haven't processed this candle yet
                        
                        if token not in last_processed_candle or last_processed_candle[token] < current_candle_end:
                            last_processed_candle[token] = current_candle_end
                            process_token_data(token)
                            # Add the logic to process the resampled data here
                            print(f"[{pd.Timestamp.now()}] Processing token: {token}")

            # Sleep briefly to yield control and avoid busy-waiting
            await asyncio.sleep(.01)  # Adjust the sleep duration as needed


In [ ]:
def process_token_data(token):
    global state 
    
    df = resampled_data[token]

    if df is None or len(df) < range_length:
        print(f"Not enough data or token {token} not found. Need at least 10 candles.")
        return  # Exit early if data is insufficient
    
    try:
        # Convert the relevant columns to NumPy arrays
        high = df['high'].values
        low = df['low'].values
        average_height, current_candle_height = calculate_average_height_range(high, low, range_length)


        # Get the current time
        current_time = pd.Timestamp.now()
        #Get the end of the last completed candle
        new_candle_start_time = current_time.floor(resample_freq)
        # Calculate the start of the last completed candle
        last_candle_start_time = new_candle_start_time - resample_freq

        # Check if the new forming candle start time is in the DataFrame
        if new_candle_start_time in df.index:
            completed_candles = df.loc[:new_candle_start_time - pd.Timedelta(microseconds=1)] # Subtract 1 nanosecond
        else:
        # If the new candle start time is not in the DataFrame, use all candles (none are forming)
            completed_candles = df
        # try:
        #     # Use get_loc to find the position (faster than 'in')
        #     _ = df.index.get_loc(new_candle_start_time)
        #     completed_candles = df.loc[:new_candle_start_time - pd.Timedelta(microseconds=1)]  # Subtract 1 microsecond
        # except KeyError:
        #     # If the new candle start time is not in the DataFrame, use all candles (none are forming)
        #     completed_candles = df

        # Extract the last two values of the indicators from the completed candles
        current_direction = completed_candles['ce_direction'].iloc[-1]
        previous_direction = completed_candles['ce_direction'].iloc[-2]

        print(f"Token: {token}, Current Direction: {current_direction}, Previous Direction: {previous_direction}")

        # Exit Logic (Prioritized)
        if current_position is not None:
            if (current_position == "call buy" and current_direction == -1 and previous_direction == 1) or \
            (current_position == "put buy" and current_direction == 1 and previous_direction == -1):
                exit_trade(state.ce_trading_token if current_position == "call buy" else state.pe_trading_token) 
                print(f"[{time.strftime('%H:%M:%S.%f', time.localtime(time.time()))[:-3]}] Exited trade for {token}")
                current_position = None





        

    except KeyError as e:
        print(f"Error processing token {token}: Column {e} not found.")
    except IndexError:
        print(f"Error processing token {token}: Not enough candles to calculate indicators.")
        

In [ ]:
def exit_trade():
    pass

In [ ]:
symbol = 'BANKNIFTY'
class TradingState:
    def __init__(self):
        self.ce_trading_symbol = None
        self.pe_trading_symbol = None
        self.ce_trading_token = None
        self.pe_trading_token = None

state = TradingState()


def find_atm_strike_and_symbols():   
    global  atm_strike
    cmp_bn = float(resampled_data[Initial_token]['close'].iloc[-1])      
    
    
    mod = int(cmp_bn) % 100
    cmp_bn = int(cmp_bn)
    atm_strike = cmp_bn - mod if mod <= 50 else cmp_bn + (100 - mod)
    #print(atm_strike)

    if Exchange == 'NSE':
        # Assuming fno_scrips_mcx is a global DataFrame
        filtered_df = fno_scrips[
            (fno_scrips['Symbol'] == symbol) &
            (fno_scrips['StrikePrice'] == float(atm_strike))
        ]
    else:
        filtered_df = fno_scrips_mcx[
            (fno_scrips_mcx['Symbol'] == symbol) &
            (fno_scrips_mcx['StrikePrice'] == float(atm_strike))
        ]

    
    if filtered_df.empty:
        print(f"Could not find trading symbols for ATM strike {atm_strike}")
        return None, None, None, None

    ce_row = filtered_df[filtered_df['OptionType'] == 'CE'].sort_values('Expiry').iloc[0]
    pe_row = filtered_df[filtered_df['OptionType'] == 'PE'].sort_values('Expiry').iloc[0]

    ce_trading_symbol, pe_trading_symbol = ce_row['TradingSymbol'], pe_row['TradingSymbol']
    ce_trading_token, pe_trading_token = ce_row['Token'], pe_row['Token']

    return ce_trading_symbol, pe_trading_symbol, ce_trading_token, pe_trading_token

async def update_atm_symbols():
    while True:
        try:
            result = await find_atm_strike_and_symbols()
            if result[0] is not None:
                (state.ce_trading_symbol, state.pe_trading_symbol, 
                 state.ce_trading_token, state.pe_trading_token) = result
                #print(f"Global symbols updated - CE: {state.ce_trading_symbol}, PE: {state.pe_trading_symbol}")
            else:
                print("Failed to update symbols")
        except Exception as e:
            print(f"Error in update_atm_symbols: {e}")
        await asyncio.sleep(5)